In [1]:
from dolfin import *

/opt/anaconda3/envs/fenicsproject/lib/python3.10/site-packages/dolfin/jit/jit.py:121: RuntimeWarning: mpi4py.MPI.Session size changed, may indicate binary incompatibility. Expected 32 from C header, got 40 from PyObject
  def compile_class(cpp_data, mpi_comm=MPI.comm_world):


In [5]:
mesh = Mesh()
with XDMFFile("non-adaptive1/mesh.xdmf") as infile:
    infile.read(mesh)

# mf = MeshFunction("bool", mesh, mesh.topology().dim())
# mf.set_all(False)

# cells = CompiledSubDomain("near(x[0], 250, tol)", tol=15)

# cells.mark(mf, True)

# with XDMFFile("non-adaptive1/mesh_marked.xdmf") as outfile:
#     outfile.write(mf)

# mesh = refine(mesh, mf)
mesh.num_vertices()*4

874216

In [4]:
with XDMFFile("non-adaptive/mesh.xdmf") as outfile:
    outfile.write(mesh)

In [7]:
import gmsh

gmsh.initialize()

gmsh.model.add("glacier_terminus")

# -------------------------------------------------------------------------
# Geometry dimensions [m]
# -------------------------------------------------------------------------

Lx = 500.0
Ly = 750.0
Lz = 125.0

# Pre-crack / notch dimensions [m]
lx = 5.0
ly = 10.0
lz = 10.0

# -------------------------------------------------------------------------
# Mesh size controls [m]
# -------------------------------------------------------------------------

h_min = 2.5
h_max = 20.0

gmsh.option.setNumber("Mesh.MeshSizeMin", h_min)
gmsh.option.setNumber("Mesh.MeshSizeMax", h_max)
gmsh.option.setNumber("Mesh.MeshSizeFactor", 1.0)

# -------------------------------------------------------------------------
# Glacier geometry
# -------------------------------------------------------------------------

glacier = gmsh.model.occ.addBox(
    0.0,
    0.0,
    0.0,
    Lx,
    Ly,
    Lz,
)

# -------------------------------------------------------------------------
# Notch geometry
# -------------------------------------------------------------------------

notch_x0 = 0.5 * Lx - 0.5 * lx
notch_y0 = 0.0
notch_z0 = Lz - lz

notch = gmsh.model.occ.addBox(
    notch_x0,
    notch_y0,
    notch_z0,
    lx,
    ly,
    lz,
)

# Subtract notch from glacier
domain, _ = gmsh.model.occ.cut(
    [(3, glacier)],
    [(3, notch)],
    removeObject=True,
    removeTool=True,
)

gmsh.model.occ.synchronize()

# -------------------------------------------------------------------------
# Internal cutting planes
#
# z = Lz/4
# z = Lz/2
# -------------------------------------------------------------------------

z1 = Lz / 4.0
z2 = Lz / 2.0

plane_z1 = gmsh.model.occ.addRectangle(
    0.0,
    0.0,
    z1,
    Lx,
    Ly,
)

plane_z2 = gmsh.model.occ.addRectangle(
    0.0,
    0.0,
    z2,
    Lx,
    Ly,
)

gmsh.model.occ.synchronize()

# -------------------------------------------------------------------------
# Fragment the glacier with the internal planes
#
# This guarantees mesh nodes at z = Lz/4 and z = Lz/2.
# -------------------------------------------------------------------------

domain, _ = gmsh.model.occ.fragment(
    domain,
    [
        (2, plane_z1),
        (2, plane_z2),
    ],
)

gmsh.model.occ.synchronize()

# -------------------------------------------------------------------------
# Physical volume
# -------------------------------------------------------------------------

volume_tags = [
    tag
    for dim, tag in domain
    if dim == 3
]

gmsh.model.addPhysicalGroup(
    3,
    volume_tags,
    1,
)

gmsh.model.setPhysicalName(
    3,
    1,
    "GLACIER",
)

# -------------------------------------------------------------------------
# Identify notch/crack surfaces
# -------------------------------------------------------------------------

tol = 1e-6
crack_surfaces = []

for dim, tag in gmsh.model.getEntities(2):
    xmin, ymin, zmin, xmax, ymax, zmax = gmsh.model.getBoundingBox(dim, tag)

    # Surfaces around the notch:
    # x approximately inside notch width,
    # y near the front boundary,
    # z near the top of the glacier.
    if (
        xmin >= notch_x0 - tol
        and xmax <= notch_x0 + lx + tol
        and ymin <= ly + tol
        and zmax >= notch_z0 - tol
    ):
        crack_surfaces.append(tag)

print("Crack surface tags:", crack_surfaces)

# -------------------------------------------------------------------------
# Local mesh refinement around crack/notch
# -------------------------------------------------------------------------

distance_field = gmsh.model.mesh.field.add("Distance")

gmsh.model.mesh.field.setNumbers(
    distance_field,
    "SurfacesList",
    crack_surfaces,
)

threshold_field = gmsh.model.mesh.field.add("Threshold")

gmsh.model.mesh.field.setNumber(
    threshold_field,
    "InField",
    distance_field,
)

# h = 2.5 m near crack
gmsh.model.mesh.field.setNumber(
    threshold_field,
    "SizeMin",
    h_min,
)

# h = 25 m far from crack
gmsh.model.mesh.field.setNumber(
    threshold_field,
    "SizeMax",
    h_max,
)

# Keep h = 2.5 m up to 10 m from the notch
gmsh.model.mesh.field.setNumber(
    threshold_field,
    "DistMin",
    10.0,
)

# Gradually increase to h = 25 m by 50 m from the notch
gmsh.model.mesh.field.setNumber(
    threshold_field,
    "DistMax",
    50.0,
)

gmsh.model.mesh.field.setAsBackgroundMesh(
    threshold_field
)

# -------------------------------------------------------------------------
# Prevent other automatic size estimates from overriding the field
# -------------------------------------------------------------------------

gmsh.option.setNumber(
    "Mesh.MeshSizeFromPoints",
    0,
)

gmsh.option.setNumber(
    "Mesh.MeshSizeFromCurvature",
    0,
)

gmsh.option.setNumber(
    "Mesh.MeshSizeExtendFromBoundary",
    0,
)

# Keep hard global limits
gmsh.option.setNumber(
    "Mesh.MeshSizeMin",
    h_min,
)

gmsh.option.setNumber(
    "Mesh.MeshSizeMax",
    h_max,
)

# -------------------------------------------------------------------------
# Generate tetrahedral mesh
# -------------------------------------------------------------------------

gmsh.model.mesh.generate(3)

gmsh.write("glacier_terminus.msh")

gmsh.finalize()

Info    : Increasing process stack size (8176 kB < 16 MB)
Crack surface tags: [34, 35, 36, 37]                                                                                                    
Info    : Meshing 1D...
Info    : [  0%] Meshing curve 38 (Line)
Info    : [ 10%] Meshing curve 39 (Line)
Info    : [ 10%] Meshing curve 40 (Line)
Info    : [ 10%] Meshing curve 41 (Line)
Info    : [ 20%] Meshing curve 42 (Line)
Info    : [ 20%] Meshing curve 43 (Line)
Info    : [ 20%] Meshing curve 44 (Line)
Info    : [ 20%] Meshing curve 45 (Line)
Info    : [ 30%] Meshing curve 46 (Line)
Info    : [ 30%] Meshing curve 47 (Line)
Info    : [ 30%] Meshing curve 48 (Line)
Info    : [ 30%] Meshing curve 49 (Line)
Info    : [ 40%] Meshing curve 50 (Line)
Info    : [ 40%] Meshing curve 51 (Line)
Info    : [ 40%] Meshing curve 52 (Line)
Info    : [ 40%] Meshing curve 53 (Line)
Info    : [ 50%] Meshing curve 54 (Line)
Info    : [ 50%] Meshing curve 55 (Line)
Info    : [ 50%] Meshing curve 56 (Line)
In

In [ ]:
import meshio

mesh = meshio.read("01_Lx500_1C_NA.msh")

cells = mesh.get_cells_type("tetra")
points = mesh.points

meshio.write("adaptive/mesh.xdmf", meshio.Mesh(
    points=points,
    cells={"tetra": cells}))

Non-Adaptive

In [6]:
import gmsh

gmsh.initialize()
gmsh.model.add("glacier_terminus")

# -------------------------------------------------------------------------
# Geometry dimensions [m]
# -------------------------------------------------------------------------

Lx = 500.0
Ly = 750.0
Lz = 125.0

# Pre-crack / notch dimensions [m]
lx = 5.0
ly = 10.0
lz = 10.0

# -------------------------------------------------------------------------
# Mesh size controls [m]
# -------------------------------------------------------------------------

h_min = 2.0
h_max = 6.15

# Refined zone size
h_refined = 2.5

# If you literally mean 2.5 mm, use:
# h_refined = 0.0025

gmsh.option.setNumber("Mesh.MeshSizeMin", h_min)
gmsh.option.setNumber("Mesh.MeshSizeMax", h_max)

# -------------------------------------------------------------------------
# Glacier geometry
# -------------------------------------------------------------------------

glacier = gmsh.model.occ.addBox(
    0.0,
    0.0,
    0.0,
    Lx,
    Ly,
    Lz,
)

# -------------------------------------------------------------------------
# Notch
# -------------------------------------------------------------------------

notch_x0 = 0.5 * Lx - 0.5 * lx
notch_y0 = 0.0
notch_z0 = Lz - lz

notch = gmsh.model.occ.addBox(
    notch_x0,
    notch_y0,
    notch_z0,
    lx,
    ly,
    lz,
)

# Subtract notch from glacier
domain, _ = gmsh.model.occ.cut(
    [(3, glacier)],
    [(3, notch)],
    removeObject=True,
    removeTool=True,
)

gmsh.model.occ.synchronize()

# -------------------------------------------------------------------------
# Physical volume
# -------------------------------------------------------------------------

volume_tags = [tag for dim, tag in domain if dim == 3]

gmsh.model.addPhysicalGroup(3, volume_tags, 1)
gmsh.model.setPhysicalName(3, 1, "GLACIER")

# -------------------------------------------------------------------------
# Local refinement zone
#
# Crack center:
#       x = 250 m
#
# Refined region:
#       x = 230 -> 270 m
#       y =   0 -> 750 m
#       z =   0 -> 125 m
#
# -------------------------------------------------------------------------

crack_center_x = 0.5 * Lx

refinement_width = 30.0

field_box = gmsh.model.mesh.field.add("Box")

gmsh.model.mesh.field.setNumber(
    field_box,
    "VIn",
    h_refined,
)

gmsh.model.mesh.field.setNumber(
    field_box,
    "VOut",
    h_max,
)

gmsh.model.mesh.field.setNumber(
    field_box,
    "XMin",
    crack_center_x - refinement_width,
)

gmsh.model.mesh.field.setNumber(
    field_box,
    "XMax",
    crack_center_x + refinement_width,
)

gmsh.model.mesh.field.setNumber(
    field_box,
    "YMin",
    0.0,
)

gmsh.model.mesh.field.setNumber(
    field_box,
    "YMax",
    Ly,
)

gmsh.model.mesh.field.setNumber(
    field_box,
    "ZMin",
    0.0,
)

gmsh.model.mesh.field.setNumber(
    field_box,
    "ZMax",
    Lz,
)

# -------------------------------------------------------------------------
# Smooth transition / threshold around the refined zone
# -------------------------------------------------------------------------

gmsh.model.mesh.field.setNumber(
    field_box,
    "Thickness",
    10.0,
)

gmsh.model.mesh.field.setAsBackgroundMesh(field_box)

# -------------------------------------------------------------------------
# Mesh settings
# -------------------------------------------------------------------------

gmsh.option.setNumber("Mesh.Algorithm3D", 10)

gmsh.option.setNumber(
    "Mesh.MeshSizeFromCurvature",
    0,
)

# Prevent point/curve/surface sizes from overriding the background field
gmsh.option.setNumber(
    "Mesh.MeshSizeFromPoints",
    0,
)

gmsh.option.setNumber(
    "Mesh.MeshSizeExtendFromBoundary",
    0,
)

# -------------------------------------------------------------------------
# Generate tetrahedral mesh
# -------------------------------------------------------------------------

gmsh.model.mesh.generate(3)

gmsh.write("glacier_terminus.msh")

gmsh.finalize()

Info    : Increasing process stack size (8176 kB < 16 MB)
Info    : Meshing 1D...                                                                                                   
Info    : [  0%] Meshing curve 13 (Line)
Info    : [ 10%] Meshing curve 14 (Line)
Info    : [ 10%] Meshing curve 15 (Line)
Info    : [ 20%] Meshing curve 16 (Line)
Info    : [ 20%] Meshing curve 17 (Line)
Info    : [ 30%] Meshing curve 18 (Line)
Info    : [ 30%] Meshing curve 19 (Line)
Info    : [ 30%] Meshing curve 20 (Line)
Info    : [ 40%] Meshing curve 21 (Line)
Info    : [ 40%] Meshing curve 23 (Line)
Info    : [ 50%] Meshing curve 24 (Line)
Info    : [ 50%] Meshing curve 25 (Line)
Info    : [ 60%] Meshing curve 26 (Line)
Info    : [ 60%] Meshing curve 27 (Line)
Info    : [ 60%] Meshing curve 28 (Line)
Info    : [ 70%] Meshing curve 29 (Line)
Info    : [ 70%] Meshing curve 30 (Line)
Info    : [ 80%] Meshing curve 31 (Line)
Info    : [ 80%] Meshing curve 32 (Line)
Info    : [ 80%] Meshing curve 33 (Line)

In [7]:
import meshio

mesh = meshio.read("glacier_terminus.msh")

cells = mesh.get_cells_type("tetra")
points = mesh.points

meshio.write("non-adaptive/mesh.xdmf", meshio.Mesh(
    points=points,
    cells={"tetra": cells}))

In [2]:
import gmsh

gmsh.initialize()
gmsh.model.add("glacier_terminus")

# -------------------------------------------------------------------------
# Geometry dimensions [m]
# -------------------------------------------------------------------------

Lx = 500.0
Ly = 750.0
Lz = 125.0

# Pre-crack / notch dimensions [m]
lx = 5.0
ly = 10.0
lz = 10.0

# -------------------------------------------------------------------------
# Mesh size controls [m]
# -------------------------------------------------------------------------

h_min = 2.0
h_max = 6.15

gmsh.option.setNumber("Mesh.MeshSizeMin", h_min)
gmsh.option.setNumber("Mesh.MeshSizeMax", h_max)

# -------------------------------------------------------------------------
# Create 3D glacier volume
# -------------------------------------------------------------------------

glacier = gmsh.model.occ.addBox(
    0.0,
    0.0,
    0.0,
    Lx,
    Ly,
    Lz,
)

# -------------------------------------------------------------------------
# Create notch
# -------------------------------------------------------------------------

notch_x0 = 0.5 * Lx - 0.5 * lx
notch_y0 = 0.0
notch_z0 = Lz - lz

notch = gmsh.model.occ.addBox(
    notch_x0,
    notch_y0,
    notch_z0,
    lx,
    ly,
    lz,
)

# -------------------------------------------------------------------------
# Subtract notch from glacier
# -------------------------------------------------------------------------

domain, _ = gmsh.model.occ.cut(
    [(3, glacier)],
    [(3, notch)],
    removeObject=True,
    removeTool=True,
)

gmsh.model.occ.synchronize()

# -------------------------------------------------------------------------
# Get all geometric curves (1D entities)
# -------------------------------------------------------------------------

curves = gmsh.model.getEntities(1)
curve_tags = [tag for dim, tag in curves]

# -------------------------------------------------------------------------
# Physical group for the 1D outline/edges
# -------------------------------------------------------------------------

if curve_tags:
    gmsh.model.addPhysicalGroup(1, curve_tags, 1)
    gmsh.model.setPhysicalName(1, 1, "OUTLINE")

# -------------------------------------------------------------------------
# Generate ONLY a 1D line mesh
# -------------------------------------------------------------------------

gmsh.model.mesh.generate(1)

# -------------------------------------------------------------------------
# Write mesh
# -------------------------------------------------------------------------

gmsh.write("glacier_terminus_1D.msh")

gmsh.finalize()

Info    : Meshing 1D...                                                                                                   
Info    : [  0%] Meshing curve 13 (Line)
Info    : [ 10%] Meshing curve 14 (Line)
Info    : [ 10%] Meshing curve 15 (Line)
Info    : [ 20%] Meshing curve 16 (Line)
Info    : [ 20%] Meshing curve 17 (Line)
Info    : [ 30%] Meshing curve 18 (Line)
Info    : [ 30%] Meshing curve 19 (Line)
Info    : [ 30%] Meshing curve 20 (Line)
Info    : [ 40%] Meshing curve 21 (Line)
Info    : [ 40%] Meshing curve 23 (Line)
Info    : [ 50%] Meshing curve 24 (Line)
Info    : [ 50%] Meshing curve 25 (Line)
Info    : [ 60%] Meshing curve 26 (Line)
Info    : [ 60%] Meshing curve 27 (Line)
Info    : [ 60%] Meshing curve 28 (Line)
Info    : [ 70%] Meshing curve 29 (Line)
Info    : [ 70%] Meshing curve 30 (Line)
Info    : [ 80%] Meshing curve 31 (Line)
Info    : [ 80%] Meshing curve 32 (Line)
Info    : [ 80%] Meshing curve 33 (Line)
Info    : [ 90%] Meshing curve 34 (Line)
Info    : [ 90%]

In [3]:
import meshio

mesh = meshio.read("glacier_terminus_1D.msh")

cells = mesh.get_cells_type("line")
points = mesh.points

meshio.write("outline.xdmf", meshio.Mesh(
    points=points,
    cells={"line": cells}))